# 第一次探索：先审计，再解释

目标：提交你自己的数据审计和一个能被实验推翻的假设。这里提供入口，把 TODO 补成你的分析。

只在本地探索行级记录；提交 notebook 前清空输出，GitHub 只保留聚合图表。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rating_lab.data import load_ratings, temporal_split
from rating_lab.models import BiasModel

ratings = load_ratings()
train, validation, test = temporal_split(ratings)
print({name: len(part) for name, part in [('train', train), ('validation', validation), ('test', test)]})

## 1. 一行是什么？

写出字段含义；时间戳记录的是哪个动作？用户至少有20条评分这一筛选会排除谁？

TODO：检查缺失、重复(user_id, movie_id)、评分范围和时间范围。不要只复制结果，解释每条检查为什么重要。

In [ ]:
# TODO: 用聚合指标完成数据审计，不要提交逐用户/逐电影明细。
# 例如 ratings.isna().sum()，并补充重复、范围和时间检查。


## 2. 时间和用户组成

TODO：在开发集（train + validation）画评分量与平均评分的时间趋势，解释均分变化的至少两个可能原因。

核查 train.max(timestamp) < validation.min(timestamp) < test.min(timestamp)。思考：随机切分会混入什么未来信息？

In [ ]:
assert train.timestamp.max() < validation.timestamp.min()
assert validation.timestamp.max() < test.timestamp.min()
# TODO: 只用 train + validation 做探索性时间图。


## 3. 冷启动和流行度

TODO：在验证集计算：新用户、新电影、两者都新，各占多少评分？再按用户等权计算，结论会变吗？

电影分组必须用 train 中的评分次数，而非全量数据。自行解释 0、1–20、21–100、>100 这四组的局限。

In [ ]:
# TODO: 使用 isin(train.user_id) 与 train.movie_id.value_counts()。
# 不要为了提高模型分数而删掉未知用户和未知电影。


## 4. 建立验证对照

下方跑一个偏置基线。TODO：比较 global、movie、user_movie，在验证集记录 RMSE/MAE 和正则化曲线。把损失函数写出来；解释每一项和未知 ID 回退。

In [ ]:
baseline = BiasModel(mode='movie', strength=5.0).fit(train)
predictions = baseline.predict(validation)
rmse = np.sqrt(np.mean((validation.rating.to_numpy() - predictions) ** 2))
print(f'Validation RMSE: {rmse:.4f}')
# TODO: 比较不同模型和 strength，记录全部候选而非只记赢家。


## 5. 交付你的解释

在 `docs/MY_FINDINGS.md` 写约300字：

1. 一个有数值支持的观察；
2. 一个机制假设和一个替代解释；
3. 一个能推翻假设的实验；
4. 数据与评估的适用范围。

初版仓库已展示测试基线。进阶模型在开发时间范围内调参，预先安排新的滚动验证或独立数据确认，避免反复看同一测试结果。